In [1]:
# Install dependencies
%pip install Anthropic python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Load environment variables from .env file
from dotenv import load_dotenv

load_dotenv()

True

In [33]:
# Create an API client
from anthropic import Anthropic
from anthropic.types import ThinkingConfigDisabledParam

client = Anthropic()
model = "claude-sonnet-4-5" # Needed for prefill support

In [31]:
# Helper functions
def add_user_message(messages: list[dict], text: str):
    messages.append({"role": "user", "content": text})

def add_assistant_message(messages: list[dict], text: str):
    messages.append({"role": "assistant", "content": text})    


def chat(messages: list[dict], 
         system_prompt:str|None=None, 
         temperature:float=1.0,
         stop_sequences:list[str]|None=None) -> str:
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }

    if system_prompt:
        params["system"] = system_prompt

    if stop_sequences:
        params["stop_sequences"] = stop_sequences
        params["thinking"] = {"type": "disabled"} # Thinking is not supported when using assistant message prefill

    return client.messages.create(**params).content[0].text   


In [5]:
# Math question example (without system prompt)
messages = []

add_user_message(messages, "How do I solve 5x + 3 = 2 for x?")
answer = chat(messages)
print(answer)


## Solving for x

**Starting equation:**
5x + 3 = 2

**Step 1: Subtract 3 from both sides**
$$5x + 3 - 3 = 2 - 3$$
$$5x = -1$$

**Step 2: Divide both sides by 5**
$$\frac{5x}{5} = \frac{-1}{5}$$
$$x = -\frac{1}{5}$$

**Answer: x = -1/5 (or -0.2)**

### Quick Check ✓
Plug x = -1/5 back into the original equation:
- 5(-1/5) + 3 = 2
- -1 + 3 = 2 ✓


In [6]:
# Math question example (with Math tutor system prompt)
messages = []
system_prompt = """
You are a patient math tutor.
Do not directly give the answer to a student's question.
Guide them to a solution step by step.
"""

add_user_message(messages, "How do I solve 5x + 3 = 2 for x?")
answer = chat(messages, system_prompt=system_prompt)
print(answer)

Let's work through this step by step!

Your goal is to **get x by itself** on one side of the equation.

**Step 1: Get rid of the +3**

What operation could you do to both sides of the equation to remove the 3 from the left side? Remember, whatever you do to one side, you must do to the other!

Give that a try and tell me what you get! 😊


In [7]:
# Concise coder example
messages = []
system_prompt = """
You are a python engineer who writes concise and efficient code.
Only write the code, do not include any explanations or comments.
Do not give options, only the most efficient solution.
"""

add_user_message(messages, "Write a Python function that checks a string for duplicate characters")
answer = chat(messages, system_prompt=system_prompt)
print(answer)

```python
def has_duplicate_chars(s: str) -> bool:
    return len(s) != len(set(s))
```


In [ ]:
# Generate two similar movie ideas by using a low temperature
messages = []
add_user_message(messages, "Generate a one sentence movie idea")
answer = chat(messages, temperature=0.0)
print(answer)

Here's a movie idea:

**A retired safecracker with early-onset dementia must break into a high-security vault before his memory fades completely — because the only cure for his condition is locked inside.


In [10]:
messages = []
add_user_message(messages, "Generate a one sentence movie idea")
answer = chat(messages, temperature=0.0)
print(answer)

Here's a movie idea:

**A retired safecracker with early-onset Alzheimer's must pull off one final heist before he forgets the combination to the vault that holds the only evidence to free his wrongfully imprisoned son.**


In [14]:
# More creative movie ideas by using a higher temperature
messages = []
add_user_message(messages, "Generate a one sentence movie idea")
answer = chat(messages, temperature=1.0)
print(answer)

Here's a movie idea:

**A washed-up lighthouse keeper discovers that the mysterious light appearing across the ocean each night is actually a signal from his deceased wife, pulling him into a dangerous obsession to reach her.


In [15]:
# Response streaming example
messages = []
add_user_message(messages, "Write a one sentence description of a fake database")

stream = client.messages.create(
        model=model,    
        max_tokens=1000,
        messages=messages,
        stream=True)

for chunk in stream:
    print(chunk)

RawMessageStartEvent(message=Message(id='msg_01VhqaJN8AZAFLkaPhsFoAmK', container=None, content=[], model='claude-sonnet-4-6', role='assistant', stop_details=None, stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=16, output_tokens=1, server_tool_use=None, service_tier='standard')), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='Here', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' is a one sentence description of a fake database:\n\n**"NutriTrack Pro', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' Database"** — A fictional nut

In [19]:
# Streaming with chunked text deltas
messages = []
add_user_message(messages, "Write a one sentence description of a fake database")

with client.messages.stream(
        model=model,    
        max_tokens=1000,
        messages=messages) as stream:
    for chunk in stream.text_stream:
        print(chunk)

    print("Streaming Done!")
    print("And we can access the full response content here:")
    print(stream.get_final_message().content[0].text)

Here
 is a one sentence description of a fake database:

**"NutriTrack Pro
 Database"** — A fictional nutritional database containing detailed
 dietary information for over 50,000 food items, including macro and micronutrient breakdowns, aller
gen flags, and glycemic index scores for common and obscure foods from around the world.
Streaming Done!
And we can access the full response content here:
Here is a one sentence description of a fake database:

**"NutriTrack Pro Database"** — A fictional nutritional database containing detailed dietary information for over 50,000 food items, including macro and micronutrient breakdowns, allergen flags, and glycemic index scores for common and obscure foods from around the world.


In [ ]:
# Event bridge rules example showing structured output
# Note this technique doesn't work for newer models that don't support assistant message prefill (e.g. sonnet-4-6+)
messages = []
add_user_message(messages, "Generate a very short event bridge rule in JSON.")
add_assistant_message(messages, "```json")
answer = chat(messages, stop_sequences=["```"])
print(answer)


{
  "source": ["aws.ec2"],
  "detail-type": ["EC2 Instance State-change Notification"],
  "detail": {
    "state": ["running"]
  }
}



In [36]:
# Structured output via Pydantic - the modern replacement for prefill on Sonnet 4.6+
# Uses client.messages.parse() with a Pydantic model as output_format.
from pydantic import BaseModel

class EventBridgeRule(BaseModel):
    source: list[str]
    detail_type: list[str]
    state: list[str]

messages = []
add_user_message(messages, "Generate a very short event bridge rule in JSON.");

response = client.messages.parse(
    model="claude-sonnet-4-6",
    max_tokens=1000,
    messages=messages,
    output_format=EventBridgeRule,
)

print(response.parsed_output)
print(type(response.parsed_output))

source=['aws.ec2'] detail_type=['EC2 Instance State-change Notification'] state=['running', 'stopped']
<class '__main__.EventBridgeRule'>


In [ ]:
# Exercise: Generate three different sample AWS CLI commands using a system prompt to specify the format. 
# Use stop sequences to ensure only the command is returned without any additional explanation.
messages = []
prompt = """
Generate three different sample AWS CLI command. Each should be very short
"""

add_user_message(messages, prompt)  
add_assistant_message(messages, "```bash")
answer = chat(messages, stop_sequences=["```"])
print(answer)


aws s3 ls

aws ec2 describe-instances

aws iam list-users

